# Práctica de NumPy aplicado a minería

Notebook de práctica personal para consolidar conceptos de NumPy usando casos aplicados a la industria minera (producción de camiones, leyes de mineral, sondajes, controles de calidad, etc).

**Objetivo:** no memorizar funciones de NumPy, sino resolver problemas concretos y entender qué herramienta usar en cada caso.


In [ ]:
import numpy as np

## Fundamentos


### Crear arrays y `dtype`

`dtype` define el tipo de dato que va a guardar el array. Es importante fijarlo explícitamente cuando el tipo por defecto que infiere NumPy no es el que necesitamos (por ejemplo, para ahorrar memoria o evitar errores de precisión).

In [ ]:
# Leyes de cobre (%) medidas en 5 sondajes exploratorios
leyes_sondajes = np.array([0.85, 1.20, 0.45, 0.92, 1.05], dtype=np.float64)

# IDs de camiones (números enteros)
ids_camiones = np.array([101, 102, 103, 104], dtype=np.int32)

# Profundidades (metros) de 4 pozos de perforación
# Nota: usar '.' como separador decimal, no ',' -> si no, NumPy lo interpreta como texto/string.
profundidad_pozo = np.array(["1.2", "1.3", "5.3", "8.3"], dtype="U32")
print(profundidad_pozo.dtype)

### Shape y Dimensions

`.shape` devuelve la forma del array (cuántos elementos hay en cada eje). `.ndim` devuelve cuántas dimensiones tiene.

In [ ]:
produccion_semanal = np.array([
    [120, 135, 140],  # Camión 1: Lun, Mar, Mié
    [110, 125, 130]   # Camión 2
])
print(produccion_semanal.shape)  # (2, 3)
print(produccion_semanal.ndim)   # 2

**Ejercicio con 3 dimensiones:** producción de camiones, por turno, por día.

Estructura: `(camiones, turnos, días)` → filas = camiones, dentro de cada camión hay turnos, y dentro de cada turno hay días.

In [ ]:
# Produccion de camiones | eje 0 = camión, eje 1 = turno, eje 2 = día
produccion_camiones = np.array([
    [
        [12, 11, 13, 14],    # turno 1
        [20, 21, 12, 12]     # turno 2
    ],  # Camion 1
    [
        [120, 121, 123, 124],  # turno 1
        [120, 121, 123, 124]   # turno 2
    ],  # Camion 2
    [
        [120, 121, 123, 124],  # turno 1
        [120, 121, 123, 124]   # turno 2
    ],  # Camion 3
])

print(produccion_camiones.shape)  # (3, 2, 4)

Recorrer el array 3D manualmente ayuda a visualizar cómo está organizado por dentro: el loop externo recorre camiones, el interno recorre turnos dentro de cada camión.

In [ ]:
for i, camion in enumerate(produccion_camiones):
    print(f"=== CAMION {i + 1} ===")
    for j, turno in enumerate(camion):
        valores_dias = ",".join(str(dia) for dia in turno)
        print(f"Turno {j + 1}: [{valores_dias}]")

### `zeros()`, `ones()`, `arange()`, `linspace()`

- `zeros(n)` / `ones(n)`: crean arrays inicializados en 0 o 1, útiles como "plantilla" a completar después.
- `arange(inicio, fin, paso)`: genera valores espaciados por un paso fijo. **El valor `fin` no se incluye.**
- `linspace(inicio, fin, cantidad)`: genera una cantidad exacta de puntos equiespaciados, **incluyendo ambos extremos**.

In [ ]:
# Registro vacío para 10 mediciones de vibración de una chancadora (a completar después)
registro_vibraciones = np.zeros(10, dtype=np.int64)
print(registro_vibraciones)

# Factor de corrección uniforme para 5 sensores
factor_correccion = np.ones(5) * 1.02

# Horas de operación registradas cada hora en un turno de 8 horas
horas_turno = np.arange(0, 8, 1)

# 5 puntos de muestreo equiespaciados a lo largo de una veta de 100 metros
puntos_muestreo = np.linspace(0, 100, 5)

**Detalle importante de `arange`:** para incluir el metro 50 en el muestreo, el límite superior tiene que ser 51 (no 50), porque `arange` excluye el último valor.

In [ ]:
# Profundidades de muestreo cada 2 metros en un sondaje de 50 metros
profundidades_de_muestreo = np.arange(0, 51, 2)
print(profundidades_de_muestreo)

**Detalle importante de `linspace` + `dtype`:** si se fuerza `dtype=int` directamente en `linspace`, los valores se **truncan** (no se redondean), y el resultado deja de estar perfectamente equiespaciado. Mejor generar los valores como float y redondear después con `np.round().astype(int)` si se necesitan enteros.

In [ ]:
# Turno de 12 horas dividido en 6 controles de calidad equiespaciados
controles_calidad = np.linspace(0, 12, 6)
print(controles_calidad)

# Versión entera, redondeando antes de convertir (mejor que forzar dtype en linspace)
controles_calidad_int = np.round(controles_calidad).astype(int)
print(controles_calidad_int)

## Manipulación de arrays


### Indexing y Slicing

Permite acceder a un elemento puntual (`indexing`) o a un subconjunto (`slicing`) del array.

In [ ]:
ley_cobre_por_zona = np.array([0.65, 0.92, 1.15, 0.48, 0.88, 1.02, 0.71])

primera_zona = ley_cobre_por_zona[0]
tres_primeras = ley_cobre_por_zona[:3]
ultimas_dos = ley_cobre_por_zona[-2:]

In [ ]:
toneladas = np.array([
    [145, 138, 120],  # Camion 1
    [152, 149, 130],  # Camion 2
    [98,  105, 90],   # Camion 3
    [160, 155, 140],  # Camion 4
    [140, 142, 125],  # Camion 5
    [155, 150, 135],  # Camion 6
    [110, 115, 100],  # Camion 7
    [148, 144, 128]   # Camion 8
])

# Toneladas del Camión 5 (índice 4) en el turno Tarde (índice 1)
turnos_camion5 = toneladas[4, 1]
print(turnos_camion5)

### Reshaping

`.reshape()` reorganiza los mismos datos en otra forma, siempre que la cantidad total de elementos coincida (por ejemplo, 24 elementos se pueden acomodar como 4x6, 2x12, 6x4, etc).

In [ ]:
# 12 mediciones de presión de una bomba, tomadas cada hora durante medio día
presion_bomba = np.arange(12)
presion_por_turno = presion_bomba.reshape(2, 6)  # 2 turnos de 6 horas

# 24 registros de temperatura de un horno, reorganizados en 4 turnos de 6 horas
registro_temperatura = np.arange(1, 25)
registro_reshape = registro_temperatura.reshape(4, 6)
print(registro_reshape)

### Flattening

`.flatten()` convierte cualquier array multidimensional en un array de una sola dimensión. Útil cuando importa el conjunto de valores, no cómo estaban organizados.

In [ ]:
matriz_leyes = np.array([[0.8, 0.9], [1.1, 0.6]])
leyes_planas = matriz_leyes.flatten()
print(f"Promedio de mediciones: {leyes_planas.mean()}")

### Transpose

`.transpose()` (o `.T`) intercambia los ejes del array. Sirve para cambiar la perspectiva de los datos sin modificar su contenido: por ejemplo, pasar de "filas = camiones, columnas = turnos" a "filas = turnos, columnas = camiones".

In [ ]:
print(toneladas.shape)  # (8, 3)

camiones_transpose = toneladas.transpose(1, 0)
print(camiones_transpose.shape)  # (3, 8)
print(camiones_transpose)

### Concatenation

`np.concatenate()` une arrays existentes en uno solo, a lo largo de un eje. Con arrays de 1D simplemente los junta en secuencia.

In [ ]:
produccion_lunes = np.array([120, 135, 140])
produccion_martes = np.array([125, 130, 145])

semana = np.concatenate([produccion_lunes, produccion_martes])
print(semana)

### Stacking

- `np.vstack()`: apila arrays como filas nuevas (crece en el eje vertical).
- `np.hstack()`: concatena arrays horizontalmente (crece en el eje horizontal, sigue siendo 1D si los inputs son 1D).

In [ ]:
turno_manana = np.array([145, 152, 98, 160])
turno_tarde = np.array([138, 149, 105, 155])

# vstack: cada array se convierte en una fila -> resultado 2D
matriz_turnos = np.vstack([turno_manana, turno_tarde])
print(matriz_turnos)

# hstack: concatena en secuencia -> resultado 1D
horizontal = np.hstack([turno_manana, turno_tarde])
print(horizontal)

## Próximos pasos

- Aplicar `sum()`, `mean()`, `argmin()`, `argmax()` sobre `produccion_camiones` (3D), usando `axis=(1,2)` para agregar por camión y `axis=(0,2)` para agregar por turno.
- Revisar la sección de agregaciones (`sum`, `mean`, `argmin`, `argmax`) — ya practicada por separado, pendiente integrarla a este mismo notebook.
